<h3> Converting TREC Qrels to JSON Format </h3>

In [21]:
import json
import os
import gzip
import logging
from collections import defaultdict

data_folder = 'trec2019-data'
qrels_filepath = os.path.join(data_folder, '2019qrels-pass.txt')
queries_filepath = os.path.join(data_folder, 'msmarco-test2019-queries.tsv.gz')
passage_filepath = os.path.join(data_folder, 'msmarco-passagetest2019-top1000.tsv.gz')

os.makedirs(data_folder, exist_ok=True)


if not os.path.exists(queries_filepath):
    logging.info("Download " + os.path.basename(queries_filepath))
    util.http_get('https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz', queries_filepath)

if not os.path.exists(qrels_filepath):
    logging.info("Download " + os.path.basename(qrels_filepath))
    util.http_get('https://trec.nist.gov/data/deep/2019qrels-pass.txt', qrels_filepath)

if not os.path.exists(passage_filepath):
    logging.info("Download " + os.path.basename(passage_filepath))
    util.http_get('https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-passagetest2019-top1000.tsv.gz', passage_filepath)


queries = {}
with gzip.open(queries_filepath, 'rt', encoding='utf8') as fIn:
    for line in fIn:
        qid, query = line.strip().split("\t")
        queries[qid] = query


relevant_docs = defaultdict(lambda: defaultdict(int))
with open(qrels_filepath) as fIn:
    for line in fIn:
        qid, _, pid, score = line.strip().split()
        score = int(score)
        if score > 0:
            relevant_docs[qid][pid] = score


relevant_qid = [qid for qid in queries if len(relevant_docs[qid]) > 0]


passage_cand = defaultdict(list)
with gzip.open(passage_filepath, 'rt', encoding='utf8') as fIn:
    for line in fIn:
        qid, pid, query, passage = line.strip().split("\t")
        passage_cand[qid].append([pid, passage])

logging.info("Queries: {}".format(len(queries)))


qrels_data = {qid: dict(relevant_docs[qid]) for qid in relevant_qid}


json_qrels_filepath = os.path.join(data_folder, 'qrels_file.json')
with open(json_qrels_filepath, 'w') as json_file:
    json.dump(qrels_data, json_file, indent=4)

logging.info(f"Qrels data saved to {json_qrels_filepath}")


<h3> Loading Run Files and Applying Fusion Methods </h3>

In [22]:
import ranx


qrels = ranx.Qrels.from_file(os.path.join(data_folder, 'qrels_file.json'), kind='json')


run1 = ranx.Run.from_file("D:\Downloads\Leiden University\Information Retrieval\IR-A1\modelsfinetuned_models\cross-encoder-cross-encoder-ms-marco-MiniLM-L-2-v2-2024-05-15_23-11-03ranking.json", kind='json')
run2 = ranx.Run.from_file("/D:\Downloads\Leiden University\Information Retrieval\IR-A1\modelsfinetuned_models\cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2024-05-16_10-29-26ranking.json", kind='json')
run3 = ranx.Run.from_file("/D:\Downloads\Leiden University\Information Retrieval\IR-A1\modelsfinetuned_models\cross-encoder-distilroberta-base-2024-05-16_12-42-14ranking.json", kind='json')

runs = [run1, run2, run3]

fusion_methods = ["sum", "mnz", "rrf", "max", "min"]

# Apply the fusion methods
results = {}
for method in fusion_methods:
    fused_run = ranx.fuse(runs, method=method)
    results[method] = fused_run
    fused_run.save(f"fused_run_{method}.json", kind='json')


<h3> Preparing Final Table </h3>

In [23]:
import pandas as pd

metrics = ["ndcg@10", "recall@100", "map@1000"]

evaluation_results = {}
for method, run in results.items():
    evaluation_results[method] = ranx.evaluate(qrels, run, metrics)

results_table = []
for method, scores in evaluation_results.items():
    row = [method]
    for metric in metrics:
        row.append(scores[metric])
    results_table.append(row)

df_results = pd.DataFrame(results_table, columns=["Fusion Method"] + metrics)

# Print the results table
print(df_results)


  Fusion Method   ndcg@10  recall@100  map@1000
0           sum  0.651970    0.511745  0.442426
1           mnz  0.651970    0.511745  0.442426
2           rrf  0.680005    0.512245  0.454345
3           max  0.617973    0.498776  0.410958
4           min  0.623737    0.434334  0.391148


<h2> Task 3 </h2>

In [24]:
import ranx
import pandas as pd

most_effective_method = "rrf"

qrels = ranx.Qrels.from_file(os.path.join(data_folder, 'qrels_file.json'), kind='json')

run1 = ranx.Run.from_file("D:\Downloads\Leiden University\Information Retrieval\IR-A1\modelsfinetuned_models\cross-encoder-cross-encoder-ms-marco-MiniLM-L-2-v2-2024-05-15_23-11-03ranking.json", kind='json')
run2 = ranx.Run.from_file("D:\Downloads\Leiden University\Information Retrieval\IR-A1\modelsfinetuned_models\cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2024-05-16_10-29-26ranking.json", kind='json')
run3 = ranx.Run.from_file("D:\Downloads\Leiden University\Information Retrieval\IR-A1\modelsfinetuned_models\cross-encoder-distilroberta-base-2024-05-16_12-42-14ranking.json", kind='json')


runs = [run1, run2, run3]

model_pairs = [
    ("model1", run1, "model2", run2),
    ("model1", run1, "model3", run3),
    ("model2", run2, "model3", run3)
]


pair_results = {}
for (name1, run1, name2, run2) in model_pairs:
    fused_run = ranx.fuse([run1, run2], method=most_effective_method)
    fused_run.save(f"fused_run_{name1}_{name2}_{most_effective_method}.json", kind='json')
    pair_results[f"{name1}_{name2}"] = fused_run

metrics = ["ndcg@10", "recall@100", "map@1000"]


evaluation_results = {}
for pair, run in pair_results.items():
    evaluation_results[pair] = ranx.evaluate(qrels, run, metrics)

results_table = []
for pair, scores in evaluation_results.items():
    row = [pair]
    for metric in metrics:
        row.append(scores[metric])
    results_table.append(row)

df_results = pd.DataFrame(results_table, columns=["Model Pair"] + metrics)

print(df_results)


      Model Pair   ndcg@10  recall@100  map@1000
0  model1_model2  0.699651    0.513374  0.462027
1  model1_model3  0.641575    0.500759  0.422999
2  model2_model3  0.648663    0.499433  0.429227
